# 3B PageRank

De oprichters van Google, Sergey Brin en Larry Page, hebben de eerste versie van hun zoekmachine gebaseerd op de Pagerank (waarbij Page eigenlijk staat voor de naam van de uitvinder eerder dan een 'web page'). Deze gebruikt de webgraaf die aanduidt welke websites met elkaar verbonden zijn via hyperlinks.

Deze graaf, met als elementen 0 als website i niet verbonden is met website j en 1 als die wel verbonden zijn, wordt omgezet in een matrix van probabiliteiten/kansen om van de ene website naar de andere te surfen om zo iteratief te berekenen welke websites het meest zullen bezocht worden. Dit laatste wordt gevat als een 'rank' of Pagerank. 

De berekening is als volgt:
- We maken eerst de kansmatrix $P$ uit de webgraaf, door de 1'en per rij te tellen en te delen door de lengte van de rij. Dan hebben we dus per rij kansen die samen tot 1 sommeren om aan te duiden met welke kans een surfer op een andere website terecht komt vanuit de huidige.
- Daarna voegen we het concept teleportatie toe: soms zullen surfers gewoon een website bezoeken zonder een link te volgen. Dit wordt toegevoegd via de `demp_factor` $df$ die de gelijke kansen uit de matrix `teleport` wat verdeelt over alle mogelijke websites om de harde 0/1-kansen te counteren. We berekenen hiervoor $H= df * P + (1-df) * teleport$.
- Daarna gaan we een initiële rank-vector (even lang als het aantal rijen/websites) vermenigvuldigen met de kans-matrix $H$ om het webgedrag te simuleren. We doen dit om de rank-waarden te laten convergeren en zo te zien welke websites het belangrijkste/meest bezocht zijn in de webgraaf. Eens de uiteindelijke `rank`-vector niet meer dan `threshold` verandert hebben we de webgraaf succesvol gesimuleerd en de ranks berekend.

Let op: voeg GEEN Jupyter notebook-cellen bij en verwijder GEEN cellen in deze notebook, anders kan je oplossing niet verbeterd worden (dit geldt **ook** voor het examen). Je moet dus enkel de functie `bereken_pagerank()` verder uitbreiden met jouw code. De `print`-statements worden niet gecontroleerd en mag je aanpassen/bijvoegen, etc.

In [1]:
import numpy as np

def bereken_pagerank(input_matrix, demp_factor=0.8, threshold=1e-8):
    P = np.zeros_like(input_matrix)

    # P[i, j] is de kans om van pagina i naar pagina j te gaan
    for j in range(N):       
        out_links = np.sum(input_matrix[:, j]) 
        if out_links == 0:
            P[:, j] = 1.0 / N
        else:
            P[:, j] = input_matrix[:, j] / out_links
    
    print("\nDe transitie-matrix P:")
    print(P)

    # teleport-matrix
    teleport = np.full((N, N), 1.0 / N)
    print("Teleportatie:\n", teleport)

    # bereken de H-matrix
    H = demp_factor * P + (1 - demp_factor) * teleport
    
    print("\nPageRank matrix H (met teleportatie):")
    print(H)

    # houd hier het resultaat in bij
    rank = np.full((N,), 1.0 / N)
    print('Leeg resultaat:', rank)
    
    # iterereer tot rank niet meer dan threshold verandert
    for it in range(1, 100):
        new_rank = H @ rank         
        if it < 3:
            print(f'Nieuwe rank in iteratie {it}\n', new_rank)
        delta = np.linalg.norm(new_rank - rank, ord=1)
        if it < 3:
            print(f'Totaal verschil tussen nieuwe rank en oude rank in iteratie {it}', delta)
        rank = new_rank
        if delta < threshold:    
            print(f"\nConvergentie na {it} iteraties met verschil ", delta)
            break
        if it % 10 == 0:
            print(f'Tussenresultaat na {it} iteraties:', rank)

    print('Rank:', rank)
    return rank

np.random.seed(10)
web_graaf = np.array([
    [0,  1,  0,  0],   
    [0,  0,  1,  0],  
    [1,  0,  0,  1],   
    [0,  0,  1,  0],   
], dtype=float)
N = web_graaf.shape[0]

rank_resultaten = bereken_pagerank(web_graaf)

print("\nUiteindelijke PageRank scores:")
for i, r in enumerate(rank_resultaten):
    print(f"  Page {i:1d}: {r:.4f}")


De transitie-matrix P:
[[0.  1.  0.  0. ]
 [0.  0.  0.5 0. ]
 [1.  0.  0.  1. ]
 [0.  0.  0.5 0. ]]
Teleportatie:
 [[0.25 0.25 0.25 0.25]
 [0.25 0.25 0.25 0.25]
 [0.25 0.25 0.25 0.25]
 [0.25 0.25 0.25 0.25]]

PageRank matrix H (met teleportatie):
[[0.05 0.85 0.05 0.05]
 [0.05 0.05 0.45 0.05]
 [0.85 0.05 0.05 0.85]
 [0.05 0.05 0.45 0.05]]
Leeg resultaat: [0.25 0.25 0.25 0.25]
Nieuwe rank in iteratie 1
 [0.25 0.15 0.45 0.15]
Totaal verschil tussen nieuwe rank en oude rank in iteratie 1 0.4
Nieuwe rank in iteratie 2
 [0.17 0.23 0.37 0.23]
Totaal verschil tussen nieuwe rank en oude rank in iteratie 2 0.3200000000000002
Tussenresultaat na 10 iteraties: [0.21182098 0.20311508 0.38194885 0.20311508]
Tussenresultaat na 20 iteraties: [0.21226373 0.20282966 0.38207696 0.20282966]
Tussenresultaat na 30 iteraties: [0.21226416 0.20283019 0.38207547 0.20283019]

Convergentie na 32 iteraties met verschil  6.04462910280823e-09
Rank: [0.21226415 0.20283019 0.38207547 0.20283019]

Uiteindelijke PageRan

### Marketingvoorbeeld: klantendoorverwijzingen

Hetzelfde wiskundige model kan ook buiten het web gebruikt worden. Stel dat we geen websites hebben, maar 4 merken (bv. lokale winkels of webshops) die aan elkaar klanten doorverwijzen: "Wie bij ons koopt, verwijzen we ook door naar...". Een pijl van merk $i$ naar merk $j$ betekent dus dat merk $i$ een deel van zijn klanten doorstuurt naar merk $j$ (bv. via een samenwerking, cross-promotie, of een aanbevelingsprogramma).

De graaf blijft identiek aan Scenario 1:
- Merk A (0) verwijst door naar Merk B (1)
- Merk B (1) verwijst door naar Merk C (2)
- Merk C (2) verwijst door naar Merk A (0)
- Merk C (2) verwijst door naar Merk D (3)
- Merk D (3) verwijst door naar Merk C (2)

In plaats van "welke website wordt het meest bezocht", meten we nu "welk merk krijgt via het doorverwijsnetwerk de meeste klantaandacht/omzet toegeschoven". De teleportatie krijgt hier ook een marketing-invulling: niet elke klant komt via een doorverwijzing binnen, sommigen vinden een merk toevallig (advertentie, toeval, mond-tot-mondreclame buiten het netwerk) - dat is precies wat de `demp_factor` $df$ modelleert.

Omdat de graaf en de `demp_factor` ongewijzigd blijven, komen de scores exact overeen met Scenario 1: Merk C krijgt de hoogste "referral rank", en is dus het merk dat het meest profiteert van het doorverwijsnetwerk.

In [4]:
np.random.seed(10)
merken = ["Merk A", "Merk B", "Merk C", "Merk D"]

# doorverwijs-netwerk: rij i, kolom j == 1 als merk i klanten doorverwijst naar merk j
doorverwijs_netwerk = np.array([
    [0,  1,  0,  0],
    [0,  0,  1,  0],
    [1,  0,  0,  1],
    [0,  0,  1,  0],
], dtype=float)
N = doorverwijs_netwerk.shape[0]

referral_rank = bereken_pagerank(doorverwijs_netwerk)

print("\nReferral rank per merk (aandeel klantaandacht uit het netwerk):")
for naam, score in zip(merken, referral_rank):
    print(f"  {naam}: {score:.4f}")


De transitie-matrix P:
[[0.  1.  0.  0. ]
 [0.  0.  0.5 0. ]
 [1.  0.  0.  1. ]
 [0.  0.  0.5 0. ]]
Teleportatie:
 [[0.25 0.25 0.25 0.25]
 [0.25 0.25 0.25 0.25]
 [0.25 0.25 0.25 0.25]
 [0.25 0.25 0.25 0.25]]

PageRank matrix H (met teleportatie):
[[0.05 0.85 0.05 0.05]
 [0.05 0.05 0.45 0.05]
 [0.85 0.05 0.05 0.85]
 [0.05 0.05 0.45 0.05]]
Leeg resultaat: [0.25 0.25 0.25 0.25]
Nieuwe rank in iteratie 1
 [0.25 0.15 0.45 0.15]
Totaal verschil tussen nieuwe rank en oude rank in iteratie 1 0.4
Nieuwe rank in iteratie 2
 [0.17 0.23 0.37 0.23]
Totaal verschil tussen nieuwe rank en oude rank in iteratie 2 0.3200000000000002
Tussenresultaat na 10 iteraties: [0.21182098 0.20311508 0.38194885 0.20311508]
Tussenresultaat na 20 iteraties: [0.21226373 0.20282966 0.38207696 0.20282966]
Tussenresultaat na 30 iteraties: [0.21226416 0.20283019 0.38207547 0.20283019]

Convergentie na 32 iteraties met verschil  6.04462910280823e-09
Rank: [0.21226415 0.20283019 0.38207547 0.20283019]

Referral rank per mer